## 1.1 安裝套件(若在colab訓練每次都需要執行)

In [ ]:
!pip install fastbook -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 預測

In [1]:
## 模型位置
from fastbook import *
from fastai.vision.widgets import *
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

## 2.1 讀取先前訓練好的權重

In [10]:
path = Path('datasets/Human_Face_Emotions/')

# View all files in directory
path.ls()

[Path('datasets/Human_Face_Emotions/Angry'), Path('datasets/Human_Face_Emotions/Fear'), Path('datasets/Human_Face_Emotions/Happy'), Path('datasets/Human_Face_Emotions/Sad'), Path('datasets/Human_Face_Emotions/Suprise')]

In [11]:
dataset = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items = get_image_files,
    splitter = RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms= Resize(224),
    # item_tfms=Resize(224, ResizeMethod.Pad, pad_mode='zeros'),
    # batch_tfms=aug_transforms(
    #           mult = 1,
    #           do_flip = True,
    #           flip_vert = True,
    #           max_rotate = 10,
    #           min_zoom = 1,
    #           max_zoom = 1,
    #           # max_lighting = 0.2,
    #           max_warp = 0,
    #           # p_affine = 0.75,
    #           # p_lighting = 0.75,
    #           # xtra_tfms = NULL,
    #           # size = NULL,
    #           mode = "bilinear",
    #           # pad_mode = "reflection",
    #           # align_corners = True,
    #           # batch = False,
    #           # min_scale = 1
    #         ),
    # get_y = parent_label
    # )
    # 自訂義
    batch_tfms = [
        *aug_transforms(
            mult=1.2,
            flip_vert=False,
            max_rotate=15,
            max_zoom=1.1,
            max_lighting=0.2,
            max_warp=0.1
        ),
        Normalize.from_stats(*imagenet_stats)
        ]
  )

#利用框架正式讀取資料
# dls = dataset.dataloaders(path, bs=16, num_workers=16)
# 自訂義
dls = ImageDataLoaders.from_folder(
    path,
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(224),
    bs=32
)

#讀取結果
print(f'此資料集共有{dls.c}種類別')
print(f'類別名稱為{dls.vocab}')
print(f'訓練資料有: {len(dls.train_ds)}筆, 驗證資料有: {len(dls.valid_ds)}筆')

此資料集共有5種類別
類別名稱為['Angry', 'Fear', 'Happy', 'Sad', 'Suprise']
訓練資料有: 47278筆, 驗證資料有: 11819筆


In [12]:
# learn = vision_learner(dls, models.resnet50, metrics=[accuracy, error_rate], pretrained=True)

# 自訂義
learn = vision_learner(
    dls,
    models.resnet50,
    pretrained=True,
    loss_func=LabelSmoothingCrossEntropy(),
    metrics=[accuracy, error_rate]
)

learn.to_fp16()

# learn.fine_tune(10)

In [19]:
myModel='./demo_stage-3.pkl'
learn = load_learner(myModel)

FileNotFoundError: [Errno 2] No such file or directory: './demo_stage-3.pkl'

## 2.2 讀取檔案並送入模型預測

In [24]:
target_cls = "Fear"

In [25]:
## 執行預測 - method I
fnames_target_cls = get_image_files(f'datasets/Human_Face_Emotions/{target_cls}')

In [26]:
fnames_target_cls

(#9732) [Path('datasets/Human_Face_Emotions/Fear/100075832.png'), Path('datasets/Human_Face_Emotions/Fear/10010.png'), Path('datasets/Human_Face_Emotions/Fear/10012.png'), Path('datasets/Human_Face_Emotions/Fear/10015.png'), Path('datasets/Human_Face_Emotions/Fear/100166637.png'), Path('datasets/Human_Face_Emotions/Fear/10025.png'), Path('datasets/Human_Face_Emotions/Fear/10029.png'), Path('datasets/Human_Face_Emotions/Fear/10032.png'), Path('datasets/Human_Face_Emotions/Fear/10040.png'), Path('datasets/Human_Face_Emotions/Fear/100420126.png'), Path('datasets/Human_Face_Emotions/Fear/10043.png'), Path('datasets/Human_Face_Emotions/Fear/10044.png'), Path('datasets/Human_Face_Emotions/Fear/10047.png'), Path('datasets/Human_Face_Emotions/Fear/100537302.png'), Path('datasets/Human_Face_Emotions/Fear/10055.png'), Path('datasets/Human_Face_Emotions/Fear/10055771.png'), Path('datasets/Human_Face_Emotions/Fear/10067.png'), Path('datasets/Human_Face_Emotions/Fear/10072.png'), Path('datasets/Hum

### 自行輸入要看哪張照片的結果

In [27]:
idx = 10

pred_class,pred_idx,outputs = learn.predict(fnames_target_cls[idx])
print(f"Actual: {target_cls}, Predicted = {pred_class}")

Actual: Fear, Predicted = Fear
